In [4]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import joblib

iris = load_iris()

data = pd.DataFrame(iris.data, columns=iris.feature_names)
data["species"] = iris.target

# Save dataset for DVC later
data.to_csv("iris.csv", index=False)

X = data.drop("species", axis=1)
y = data["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

joblib.dump(model, "model.pkl")

print("Model trained and saved successfully")


ModuleNotFoundError: No module named 'pandas'

In [ ]:
#app.py
from flask import Flask, render_template, request
import joblib

app = Flask(__name__)
model = joblib.load("model.pkl")

species_map = {0:"Setosa",1:"Versicolor",2:"Virginica"}

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():

    features = [
        float(request.form["sepal_length"]),
        float(request.form["sepal_width"]),
        float(request.form["petal_length"]),
        float(request.form["petal_width"])
    ]

    prediction = model.predict([features])[0]
    result = species_map[prediction]

    return render_template("index.html", prediction_text=result)

if __name__ == "__main__":
    app.run(debug=True)


In [ ]:
#index.html
<!DOCTYPE html>
<html>
<head>
<title>Iris Classifier</title>
<link rel="stylesheet" href="/static/style.css">
</head>
<body>

<div class="container">
<h1>Iris Flower Classifier</h1>

<form action="/predict" method="post">

<input type="number" step="0.1" name="sepal_length" placeholder="Sepal Length">
<input type="number" step="0.1" name="sepal_width" placeholder="Sepal Width">
<input type="number" step="0.1" name="petal_length" placeholder="Petal Length">
<input type="number" step="0.1" name="petal_width" placeholder="Petal Width">

<button type="submit">Predict</button>

</form>

{% if prediction_text %}
<h2>{{ prediction_text }}</h2>
{% endif %}

</div>

</body>
</html>


In [ ]:
#style.css
body { font-family: Arial; background: #f4f6f8; }
.container { width: 400px; margin: 60px auto; text-align: center; }
input { display: block; margin: 10px auto; padding: 10px; }
button { padding: 10px 20px; background: green; color: white; border: none; }


In [ ]:
#Modified
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import joblib
import mlflow

iris = load_iris()

data = pd.DataFrame(iris.data, columns=iris.feature_names)
data["species"] = iris.target

# Save dataset for DVC later
data.to_csv("iris.csv", index=False)

X = data.drop("species", axis=1)
y = data["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

with mlflow.start_run():
    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)

joblib.dump(model, "model.pkl")

print("Model trained and saved successfully")


In [ ]:
#Updated code
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import joblib
import mlflow

# 1. Load Data
iris = load_iris()
data = pd.DataFrame(iris.data, columns=iris.feature_names)
data["species"] = iris.target
data.to_csv("iris.csv", index=False)

X = data.drop("species", axis=1)
y = data["species"]

# 2. Split Data
test_size = 0.2
random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state
)

# 3. MLflow Tracking Block
with mlflow.start_run():
    # --- LOG PARAMETERS (Settings you chose) ---
    mlflow.log_param("test_size", test_size)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("max_iter", 200)

    # --- TRAIN MODEL ---
    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)

    # --- LOG METRICS (The results) ---
    accuracy = model.score(X_test, y_test)
    mlflow.log_metric("accuracy", accuracy)

    # --- LOG ARTIFACTS (The actual files) ---
    joblib.dump(model, "model.pkl")
    mlflow.log_artifact("model.pkl")
    mlflow.log_artifact("iris.csv")

print(f"Model trained with accuracy: {accuracy}")

Day 2 CI/CD Docker

In [ ]:
app.run(host="0.0.0.0", port=5000, debug=True)

Dockerfile

In [ ]:
FROM python:3.10

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir -r requirement.txt

CMD ["python", "app.py"]


Build docker image

In [ ]:

docker build -t iris-ml-app .


SHOW IMAGE CREATED

In [ ]:
docker images


RUN CONTAINER

In [ ]:
docker run -p 5000:5000 iris-ml-app

Save Docker Image

In [ ]:
docker save iris-ml-app > iris-ml-app.tar

#CI/CD


In [ ]:
mkdir .github
cd .github
mkdir workflows

Create Workflow File

In [ ]:
name: Iris ML CI Pipeline

on:
  push:
    branches: [ main ]

jobs:
  build:

    runs-on: ubuntu-latest

    steps:

    - name: Checkout repository
      uses: actions/checkout@v3

    - name: Setup Python
      uses: actions/setup-python@v4
      with:
        python-version: "3.10"

    - name: Install dependencies
      run: |
        pip install -r requirements.txt

    - name: Run training script
      run: python train.py

    - name: Verify model exists
      run: |
        if [ ! -f model.pkl ]; then
          echo "Model not created"
          exit 1
        fi

    - name: Upload model artifact
      uses: actions/upload-artifact@v4
      with:
        name: trained-model
        path: model.pkl


In [ ]:
git add .
git commit -m "Add CI pipeline"
git push